# 🔢 Linear Algebra for Machine Learning — Hands-On

Every ML model is a sequence of linear algebra operations. This notebook builds from vectors to SVD/PCA with explicit ML connections at every step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=3, suppress=True)
print('Ready!')

---
## 1. Scalars, Vectors, Matrices, Tensors

In [ ]:
# Scalar
scalar = 3.14
print(f"Scalar: {scalar}  (shape: {np.array(scalar).shape})")

# Vector — one data sample with 4 features
vector = np.array([1.2, 3.5, -0.7, 2.1])
print(f"Vector: {vector}  shape={vector.shape}")

# Matrix — dataset of 3 samples, 4 features each
matrix = np.array([[1.2, 3.5, -0.7, 2.1],
                   [0.5, 1.2,  0.3, 4.0],
                   [3.1, 0.0, -1.2, 0.8]])
print(f"Matrix shape: {matrix.shape}  (3 samples × 4 features)")

# Tensor — batch of 32 RGB images 28×28
import torch
image_batch = torch.zeros(32, 3, 28, 28)   # (batch, channels, height, width)
print(f"Image batch tensor shape: {image_batch.shape}")

# Attention scores tensor in Transformer
attn = torch.zeros(8, 12, 512, 512)  # (batch, heads, seq_len, seq_len)
print(f"Attention tensor shape:   {attn.shape}  (batch × heads × seq × seq)")

---
## 2. Vector Geometry — Visualising Data as Points

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Vector addition ---
u = np.array([2, 1])
v = np.array([1, 2])
w = u + v

ax = axes[0]
ax.quiver(0,0, u[0],u[1], angles='xy', scale_units='xy', scale=1, color='blue', label='u')
ax.quiver(u[0],u[1], v[0],v[1], angles='xy', scale_units='xy', scale=1, color='red', label='v (from u tip)')
ax.quiver(0,0, w[0],w[1], angles='xy', scale_units='xy', scale=1, color='green', label='u+v')
ax.set_xlim(-1, 5); ax.set_ylim(-1, 5)
ax.set_aspect('equal')
ax.set_title('Vector Addition: Tip-to-Tail')
ax.legend()
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)

# --- Scalar multiplication ---
ax2 = axes[1]
v0 = np.array([1, 0.5])
for scale, col, label in [(1,'blue','1·v'), (2,'red','2·v'), (-1,'green','-1·v'), (0.5,'purple','0.5·v')]:
    ax2.quiver(0,0, scale*v0[0], scale*v0[1], angles='xy', scale_units='xy', scale=1,
               color=col, label=label)
ax2.set_xlim(-2.5, 2.5); ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.set_title('Scalar Multiplication: Stretch/Flip')
ax2.legend()
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

---
## 3. Matrix Multiplication — The Core of Every Neural Layer

In [ ]:
# A linear layer: Z = XW^T + b
# X: 3 samples × 4 features
# W: 2 output neurons × 4 features
np.random.seed(0)
X = np.random.randn(3, 4)   # batch of 3, 4 features each
W = np.random.randn(2, 4)   # weight matrix: 2 neurons, 4 inputs each
b = np.array([0.1, -0.2])   # bias for 2 neurons

Z = X @ W.T + b   # shape: (3, 2)

print("Input X (3 samples × 4 features):")
print(X)
print("\nWeight W (2 neurons × 4 features):")
print(W)
print("\nOutput Z = X @ W.T + b  (3 samples × 2 outputs):")
print(Z)
print("\n💡 Each row of Z = output of 2 neurons for one sample")
print("   Z[0,0] = dot(X[0], W[0]) + b[0]  =", round(np.dot(X[0], W[0]) + b[0], 5))

# Verify shapes make sense
print(f"\nShape check: ({X.shape}) @ ({W.T.shape}) = {Z.shape}")

In [ ]:
# Matrix as a Geometric Transformation — Visualise it!
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Draw a unit square
def draw_grid(ax, M, color='steelblue', title=''):
    # Transform a grid of vectors
    pts = np.array([[x, y] for x in np.linspace(-1, 1, 5) for y in np.linspace(-1, 1, 5)])
    transformed = (M @ pts.T).T
    ax.scatter(pts[:, 0], pts[:, 1], c='lightgray', s=20, label='Original')
    ax.scatter(transformed[:, 0], transformed[:, 1], c=color, s=40, label='Transformed')
    # Draw basis vectors
    e1, e2 = np.array([1,0]), np.array([0,1])
    Me1, Me2 = M @ e1, M @ e2
    ax.quiver(0,0, Me1[0], Me1[1], color='red', scale=1, scale_units='xy', angles='xy', width=0.02)
    ax.quiver(0,0, Me2[0], Me2[1], color='green', scale=1, scale_units='xy', angles='xy', width=0.02)
    ax.set_xlim(-3,3); ax.set_ylim(-3,3)
    ax.set_aspect('equal')
    ax.axhline(0,c='k',lw=0.5); ax.axvline(0,c='k',lw=0.5)
    ax.set_title(title)

M_rotate = np.array([[0, -1], [1, 0]])    # 90° rotation
M_scale  = np.array([[2, 0], [0, 0.5]])   # scale x by 2, y by 0.5
M_shear  = np.array([[1, 1], [0, 1]])     # shear

draw_grid(axes[0], M_rotate, 'blue',      f'Rotation\ndet={np.linalg.det(M_rotate):.1f}')
draw_grid(axes[1], M_scale,  'darkorange', f'Scaling\ndet={np.linalg.det(M_scale):.1f}')
draw_grid(axes[2], M_shear,  'green',     f'Shear\ndet={np.linalg.det(M_shear):.1f}')

plt.suptitle('Matrices as Geometric Transformations (red=e₁ image, green=e₂ image)')
plt.tight_layout()
plt.show()
print("💡 Every weight matrix W in a neural network is a geometric transformation.")
print("   Deep network = composition of many such transformations, with nonlinearities in between.")

---
## 4. Dot Products — Similarity & Attention

In [ ]:
# Cosine similarity between word embeddings
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Simplified word vectors (normally 512-dim etc)
np.random.seed(1)
embeddings = {
    'king':    np.array([0.9, 0.8, 0.1, -0.2]),
    'queen':   np.array([0.8, 0.9, 0.9, -0.1]),
    'man':     np.array([0.8, 0.1, 0.0,  0.1]),
    'woman':   np.array([0.7, 0.2, 0.9,  0.0]),
    'dog':     np.array([0.1, 0.1, 0.0,  0.9]),
    'cat':     np.array([0.1, 0.0, 0.1,  0.8]),
    'computer':np.array([0.0, 0.0, 0.0, -0.8]),
}

words = list(embeddings.keys())
sim_matrix = np.zeros((len(words), len(words)))
for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        sim_matrix[i,j] = cosine_sim(embeddings[w1], embeddings[w2])

plt.figure(figsize=(8, 6))
im = plt.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(words)), words, rotation=45)
plt.yticks(range(len(words)), words)
for i in range(len(words)):
    for j in range(len(words)):
        plt.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.title('Cosine Similarity Between Word Embeddings\n(dot product normalised — measures angle between vectors)')
plt.tight_layout()
plt.show()

# Vector arithmetic
king_minus_man = embeddings['king'] - embeddings['man']
analogy = embeddings['woman'] + king_minus_man
sims = {w: cosine_sim(analogy, embeddings[w]) for w in words}
print("\nAnalogy: king - man + woman = ?")
for w, s in sorted(sims.items(), key=lambda x: -x[1]):
    print(f"  {w:12s}: {s:.3f}")

In [ ]:
# Scaled Dot-Product Attention (the heart of Transformers)
np.random.seed(42)
seq_len = 5
d_k = 8  # key/query dim

# Simulate Q, K, V matrices for a single attention head
Q = np.random.randn(seq_len, d_k)  # queries: what each token is looking for
K = np.random.randn(seq_len, d_k)  # keys:    what each token offers
V = np.random.randn(seq_len, d_k)  # values:  what each token contributes

# Attention scores
scores = Q @ K.T / np.sqrt(d_k)   # shape: (seq_len, seq_len)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # numerical stability
    return np.exp(x) / np.exp(x).sum(axis=axis, keepdims=True)

attn_weights = softmax(scores)  # probabilities summing to 1 per row
output = attn_weights @ V       # weighted sum of values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im1 = axes[0].imshow(scores, cmap='coolwarm')
axes[0].set_title(f'Raw Attention Scores = Q @ K.T / √{d_k}\n(dot product similarity)')
plt.colorbar(im1, ax=axes[0])
axes[0].set_xlabel('Key position (what token offers)'); axes[0].set_ylabel('Query position (what token seeks)')

im2 = axes[1].imshow(attn_weights, cmap='Blues')
axes[1].set_title('Attention Weights after Softmax\n(each row sums to 1 — a probability distribution)')
plt.colorbar(im2, ax=axes[1])
axes[1].set_xlabel('Source token'); axes[1].set_ylabel('Target token')

plt.tight_layout()
plt.show()
print("💡 Each row = where this token 'attends' to. Higher = more attention to that source token.")
print("   The output is a weighted average of V (values) according to these attention weights.")

---
## 5. Norms — Measuring Vectors & Regularisation

In [ ]:
# Visualise L1 vs L2 norm unit balls
theta = np.linspace(0, 2*np.pi, 1000)

# L2 unit ball: x^2 + y^2 = 1  → circle
x_l2 = np.cos(theta)
y_l2 = np.sin(theta)

# L1 unit ball: |x| + |y| = 1  → diamond
x_l1 = np.cos(theta); y_l1 = np.sin(theta)
norm_l1 = np.abs(x_l1) + np.abs(y_l1)
x_l1 /= norm_l1; y_l1 /= norm_l1

# L∞ unit ball: max(|x|,|y|) = 1  → square
x_linf = np.cos(theta); y_linf = np.sin(theta)
norm_linf = np.maximum(np.abs(x_linf), np.abs(y_linf))
x_linf /= norm_linf; y_linf /= norm_linf

plt.figure(figsize=(8, 8))
plt.plot(x_l1, y_l1, 'r-', lw=2, label='L1 norm = 1 (diamond)')
plt.plot(x_l2, y_l2, 'b-', lw=2, label='L2 norm = 1 (circle)')
plt.plot(x_linf, y_linf, 'g-', lw=2, label='L∞ norm = 1 (square)')
plt.axhline(0, c='k', lw=0.5); plt.axvline(0, c='k', lw=0.5)
plt.legend(fontsize=12)
plt.title('Unit Balls for Different Norms\n(All points at "distance" 1 from origin)')
plt.axis('equal')
plt.xlim(-1.5, 1.5); plt.ylim(-1.5, 1.5)
plt.show()

# Regularisation demo
print("\n=== Regularisation Effect on Weights ===")
from sklearn.linear_model import Ridge, Lasso
from sklearn.datasets import make_regression
np.random.seed(0)
X_reg, y_reg = make_regression(n_samples=50, n_features=20, n_informative=5, noise=10, random_state=0)

for name, model in [('No reg (OLS)', Ridge(alpha=1e-10)), ('L2 Ridge α=1', Ridge(alpha=1)), ('L1 Lasso α=1', Lasso(alpha=1))]:
    model.fit(X_reg, y_reg)
    w = model.coef_
    print(f"{name:20s}: {(w==0).sum():2d} zero weights, L1={np.abs(w).sum():.2f}, L2={np.linalg.norm(w):.2f}")

---
## 6. Eigenvalues & Eigenvectors — The Geometry of Transformations

In [ ]:
# Find and visualise eigenvectors of a 2×2 matrix
A = np.array([[3, 1],
              [1, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print("Matrix A:")
print(A)
print(f"\nEigenvalues: {eigenvalues}")
print("Eigenvectors (columns):")
print(eigenvectors)

# Verify: Av = λv
for i in range(2):
    v = eigenvectors[:, i]
    Av = A @ v
    lv = eigenvalues[i] * v
    print(f"\nEigenvector {i+1}: {v.round(4)}")
    print(f"A @ v = {Av.round(4)},  λv = {lv.round(4)}  ✓ equal")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Draw random vectors and their transformed versions
np.random.seed(3)
vecs = np.random.randn(8, 2)
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)  # normalise

ax = axes[0]
for v in vecs:
    Av = A @ v
    ax.quiver(0,0, v[0],v[1], color='blue', alpha=0.4, scale=1, scale_units='xy', angles='xy')
    ax.quiver(0,0, Av[0],Av[1], color='red', alpha=0.4, scale=1, scale_units='xy', angles='xy')

# Eigenvectors
for i in range(2):
    v = eigenvectors[:, i]
    Av = A @ v
    ax.quiver(0,0, v[0],v[1], color='darkblue', scale=1, scale_units='xy', angles='xy', width=0.02, label=f'Eigvec {i+1}')
    ax.quiver(0,0, Av[0],Av[1], color='darkred', scale=1, scale_units='xy', angles='xy', width=0.02)

ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.set_aspect('equal')
ax.axhline(0,c='k',lw=0.5); ax.axvline(0,c='k',lw=0.5)
ax.set_title('Blue vectors → Red vectors (after A)\n(Eigenvectors only get scaled, not rotated)')
ax.legend()

# Vanishing/Exploding gradients in RNN
ax2 = axes[1]
T = 50  # time steps
for eig_max, col, label in [(0.9, 'blue', '|λ|=0.9 (vanish)'), (1.0, 'green', '|λ|=1.0 (stable)'), (1.1, 'red', '|λ|=1.1 (explode)')]:
    W_rnn = np.array([[eig_max]])
    h = 1.0
    hs = [h]
    for _ in range(T):
        h = W_rnn[0,0] * h
        hs.append(h)
    ax2.plot(hs[:30], color=col, label=label, lw=2)
ax2.set_title('RNN Gradient Signal Through Time\n(= repeated multiplication by eigenvalue)')
ax2.set_xlabel('Time steps'); ax2.set_ylabel('Gradient magnitude')
ax2.legend()
ax2.set_ylim(-0.5, 5)

plt.tight_layout()
plt.show()
print("💡 RNNs vanish/explode because gradients = eigenvalue^T.")
print("   LSTM's gating mechanism keeps |λ| ≈ 1 for relevant time steps.")

---
## 7. Singular Value Decomposition (SVD) — The Grand Unification

In [ ]:
# Image compression via SVD
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[0]  # 8×8 grayscale image

# Full SVD
U, S, Vt = np.linalg.svd(img, full_matrices=False)

print(f"Original image shape: {img.shape}")
print(f"U: {U.shape}, S: {S.shape}, Vt: {Vt.shape}")
print(f"Singular values: {S.round(2)}")
print(f"\nNum params original: {img.size}")

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')

for ax, k in zip(axes[1:], [1, 2, 4, 8]):
    # Rank-k approximation: sum of k outer products
    approx = sum(S[i] * np.outer(U[:,i], Vt[i,:]) for i in range(k))
    params = k * (U.shape[0] + 1 + Vt.shape[1])
    ax.imshow(approx, cmap='gray')
    ax.set_title(f'Rank-{k}\n{params} params')
    ax.axis('off')

plt.suptitle(f'SVD Image Compression (original = {img.size} values)', y=1.02)
plt.tight_layout()
plt.show()

# Variance explained plot
var_explained = S**2 / np.sum(S**2)
cumvar = np.cumsum(var_explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(S)), var_explained * 100, color='steelblue')
axes[0].set_xlabel('Singular value index')
axes[0].set_ylabel('% variance explained')
axes[0].set_title('Variance per Singular Value')

axes[1].plot(cumvar * 100, 'o-', color='darkorange')
axes[1].axhline(95, color='red', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Rank k'); axes[1].set_ylabel('Cumulative % variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# LoRA (Low-Rank Adaptation) — how SVD enables efficient LLM fine-tuning
print("=" * 60)
print("LoRA: Low-Rank Adaptation of LLMs")
print("=" * 60)

d = 512  # hypothetical weight matrix dimension
r = 8    # low-rank dimension

full_params = d * d
lora_params = d * r + r * d  # A (d×r) + B (r×d)

print(f"\nFull weight matrix W ∈ R^{d}×{d}: {full_params:,} parameters")
print(f"LoRA decomposition W ≈ W₀ + A @ B:")
print(f"  A ∈ R^{d}×{r}: {d*r:,} params")
print(f"  B ∈ R^{r}×{d}: {r*d:,} params")
print(f"  Total LoRA params: {lora_params:,}")
print(f"  Compression ratio: {full_params/lora_params:.1f}x fewer parameters!")
print()
print("Why does this work? If the weight updates during fine-tuning")
print("are approximately low-rank (intrinsic dimensionality is small),")
print("then ΔW ≈ AB is a good approximation with far fewer parameters.")

# Simulate: is the update low-rank?
np.random.seed(0)
W_pretrained = np.random.randn(64, 64) * 0.1
# Fine-tuned weight is close to pretrained with a low-rank perturbation
A_true = np.random.randn(64, 4) * 0.1
B_true = np.random.randn(4, 64) * 0.1
W_finetuned = W_pretrained + A_true @ B_true + np.random.randn(64,64)*0.001  # tiny noise

delta_W = W_finetuned - W_pretrained
_, S_delta, _ = np.linalg.svd(delta_W)

plt.figure(figsize=(8, 4))
plt.bar(range(len(S_delta)), S_delta, color='purple', alpha=0.7)
plt.xlabel('Singular value index')
plt.ylabel('Magnitude')
plt.title('Singular Values of Weight Update ΔW\n(First 4 dominate → ΔW is approximately rank-4 → LoRA works!)')
plt.show()

---
## 8. Principal Component Analysis (PCA) — From Scratch

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Load Iris dataset (4D)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
labels = iris.target_names

# ── PCA from scratch ──
# Step 1: Centre the data
X_centred = X_iris - X_iris.mean(axis=0)

# Step 2: Covariance matrix
C = (X_centred.T @ X_centred) / (len(X_centred) - 1)  # (4, 4)

# Step 3: Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(C)         # eigh for symmetric
# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Step 4: Project to top-2 PCs
Z_scratch = X_centred @ eigenvectors[:, :2]

# Verify against sklearn
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
Z_sklearn = pca.fit_transform(X_iris)

print("PCA from scratch vs sklearn:")
print(f"Max difference: {np.abs(np.abs(Z_scratch) - np.abs(Z_sklearn)).max():.1e}  ✓")
print(f"\nExplained variance ratio: {eigenvalues/eigenvalues.sum()}")
print(f"Top 2 PCs explain {eigenvalues[:2].sum()/eigenvalues.sum():.1%} of variance")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['blue', 'red', 'green']
for i, (cls, col) in enumerate(zip(labels, colors)):
    mask = y_iris == i
    axes[0].scatter(Z_scratch[mask, 0], Z_scratch[mask, 1], c=col, label=cls, alpha=0.7, s=60)
    axes[1].scatter(X_iris[mask, 0], X_iris[mask, 2], c=col, label=cls, alpha=0.7, s=60)

axes[0].set_xlabel(f'PC1 ({eigenvalues[0]/eigenvalues.sum():.1%} var)')
axes[0].set_ylabel(f'PC2 ({eigenvalues[1]/eigenvalues.sum():.1%} var)')
axes[0].set_title('Iris Data in PCA Space (2D from 4D)\nClasses are well-separated!')
axes[0].legend()

axes[1].set_xlabel('Sepal length (feature 1)')
axes[1].set_ylabel('Petal length (feature 3)')
axes[1].set_title('Original Feature Space (just 2 of 4 features)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Scree plot
explained = eigenvalues / eigenvalues.sum()
plt.figure(figsize=(6, 4))
plt.bar(range(1, 5), explained * 100, color='steelblue', label='Individual')
plt.plot(range(1, 5), np.cumsum(explained) * 100, 'ro-', label='Cumulative')
plt.xlabel('Principal Component'); plt.ylabel('Variance Explained (%)')
plt.title('Scree Plot — How Many PCs to Keep?')
plt.legend()
plt.show()

In [ ]:
# PCA Whitening — preprocessing for ML
# Whitening: decorrelate features AND normalise variance to 1

# Original data (correlated, different scales)
np.random.seed(0)
mean = np.array([3, 5])
cov_orig = np.array([[4, 3], [3, 3]])
X_corr = np.random.multivariate_normal(mean, cov_orig, 500)

# Standardise
X_std = (X_corr - X_corr.mean(0)) / X_corr.std(0)

# PCA whitening
C = np.cov(X_std.T)
vals, vecs = np.linalg.eigh(C)
# W_white = Λ^(-1/2) Q^T
W_white = (vecs / np.sqrt(vals)) @ vecs.T
X_white = (X_std @ W_white.T)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, Xp, title in zip(axes, [X_corr, X_std, X_white],
                          ['Original (correlated, different scales)',
                           'Standardised (centred, unit std)',
                           'PCA Whitened (zero corr, unit variance)']):
    ax.scatter(Xp[:,0], Xp[:,1], alpha=0.2, s=10)
    ax.set_title(title)
    ax.set_aspect('equal')
    C_p = np.cov(Xp.T)
    ax.set_xlabel(f'Cov matrix:\n[[{C_p[0,0]:.2f},{C_p[0,1]:.2f}]\n [{C_p[1,0]:.2f},{C_p[1,1]:.2f}]]', fontsize=9)

plt.tight_layout()
plt.show()
print("💡 Whitened data has identity covariance matrix — no correlations between features.")
print("   Makes gradient descent much more efficient (isotropic loss landscape).")

In [ ]:
# Full Neural Network Forward Pass — Pure Linear Algebra
import torch
import torch.nn as nn

print("=" * 60)
print("Neural Network as Linear Algebra Operations")
print("=" * 60)

torch.manual_seed(0)

# 3-layer network
model = nn.Sequential(
    nn.Linear(4, 8),   # W1 ∈ R^(8×4), b1 ∈ R^8
    nn.ReLU(),
    nn.Linear(8, 4),   # W2 ∈ R^(4×8)
    nn.ReLU(),
    nn.Linear(4, 3),   # W3 ∈ R^(3×4)
    nn.Softmax(dim=-1)
)

x = torch.tensor(X_iris[:5], dtype=torch.float32)
out = model(x)

print(f"\nInput shape: {x.shape}  (5 flowers × 4 features)")
for name, layer in model.named_modules():
    if isinstance(layer, nn.Linear):
        print(f"  Layer {name}: W ∈ R^{layer.weight.shape}, b ∈ R^{layer.bias.shape}")
print(f"\nOutput shape: {out.shape}  (5 flowers × 3 class probabilities)")
print("\nOutput (softmax probabilities — must sum to 1):")
print(out.detach().numpy().round(3))
print(f"Row sums: {out.sum(dim=1).detach().numpy().round(4)}  ✓")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters = {total_params}")
print("All of them are elements of weight matrices and bias vectors.")
print("Training = finding the values of these matrix entries via gradient descent.")